In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from scripts.VectorFieldEmbedder import VectorFieldEmbedder

# --------------------------------------------------
# Larry: load precomputed distance-based color
# --------------------------------------------------
larry_embedding_path = "./data/real_data_benchmark/larry/flowmap_embedding.npy"
larry_color_path = "./data/real_data_benchmark/larry/color_distance_from_root.npy"

# Load embedding (used elsewhere)
X_emb = np.load(larry_embedding_path)

# Load visualization color directly
color_larry = np.load(larry_color_path)

print(
    f"Loaded Larry embedding: {X_emb.shape}\n"
    f"Loaded Larry color: {color_larry.shape}"
)

# -----------------------------
# DATASET REGISTRY (FINAL)
# -----------------------------
DATASETS = {
    "cell_cycle": {
        "X": "./data/real_data_benchmark/cell_cycle/X_cc.npy",
        "V": "./data/real_data_benchmark/cell_cycle/V_cc.npy",
        "color": "./data/real_data_benchmark/cell_cycle/color_cell_cycle_relativePos.npy",
        "embedding": "./data/real_data_benchmark/cell_cycle/flowmap_embedding.npy",
    },
    "pancreas": {
        "X": "./data/real_data_benchmark/pancreas/X_pca.npy",
        "V": "./data/real_data_benchmark/pancreas/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/pancreas/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/pancreas/scvelo_umap_embedding.npy",
    },
    "dentate_gyrus": {
        "X": "./data/real_data_benchmark/dentate_gyrus/X_pca.npy",
        "V": "./data/real_data_benchmark/dentate_gyrus/V_pca_dynamical.npy",
        "color": "./data/real_data_benchmark/dentate_gyrus/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/dentate_gyrus/scvelo_umap_embedding.npy",
    },
    "larry": {
        "X": "./data/real_data_benchmark/larry/X_raw.npy",
        "V": "./data/real_data_benchmark/larry/V_raw.npy",
        "color": "./data/real_data_benchmark/larry/distance_pseudotime.npy",
        "embedding": "./data/real_data_benchmark/larry/flowmap_embedding.npy",
    },
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import plot_velocity_streamplot


def flowmap_plot_single(
    dataset_name,
    cfg,
    stream_density=0.7,
    streamline_thickness=1.0,
    arrowsize=2.5,
    scatter_size=200,
    scatter_alpha=0.15,
):

    print(f"\n=== Running FlowMap on: {dataset_name} ===")

    # ---- Load matrices ----
    X = np.load(cfg["X"])
    V = np.load(cfg["V"])
    color = np.load(cfg["color"])          # pseudotime / phase
    embedding_init = np.load(cfg["embedding"])

    # ---- Ensure color is 1D float ----
    color = np.asarray(color, dtype=float).ravel()
    
    # ---- Mask valid points ----
    valid_mask = ~np.isnan(color)
    
    # ---- Normalize ONLY valid values ----
    if valid_mask.sum() > 0:
        c_valid = color[valid_mask]
        c_valid = (c_valid - c_valid.min()) / (c_valid.max() - c_valid.min() + 1e-8)
    
        color_norm = np.full_like(color, np.nan, dtype=float)
        color_norm[valid_mask] = c_valid
    else:
        color_norm = np.zeros_like(color)

    # ---- Run FlowMap (fixed embedding) ----
    emb = VectorFieldEmbedder(
        X,
        V,
        dist_method="phase",
        X_emb=embedding_init,
        embed_kwargs={"n_neighbors": 30, "min_dist": 0.6},
        dof=30,
        method="umap",
    )
    emb.initialize_embedding(seed=0)

    # ---- Plot ----
    plot_velocity_streamplot(
        X_2d=emb.X_emb,
        tps_vf=emb.tps_vf,
        scatter_color=color,
        cmap="viridis",
        show_colorbar=False,
        show_axes=False,
        stream_density=stream_density,
        streamline_thickness=streamline_thickness,
        arrowsize=arrowsize,
        scatter_size=scatter_size,
        scatter_alpha=scatter_alpha,
        figsize=(6, 5),
    )

    return emb

In [ ]:
flowmap_results = {}

# cell_cycle
emb_cc = flowmap_plot_single(
    "cell_cycle", 
    DATASETS["cell_cycle"],
    scatter_size=200, scatter_alpha=0.1,
    stream_density=0.5,
    streamline_thickness=7.0,
    arrowsize=3.0
)
flowmap_results["cell_cycle"] = emb_cc

In [ ]:
# pancreas
emb_pan = flowmap_plot_single(
    "pancreas",
    DATASETS["pancreas"],
    scatter_size=100, scatter_alpha=0.05,
    stream_density=1.0,
    streamline_thickness=4.0,
    arrowsize=2.0
)
flowmap_results["pancreas"] = emb_pan

In [ ]:
# dentate gyrus
emb_dg = flowmap_plot_single(
    "dentate_gyrus",
    DATASETS["dentate_gyrus"],
    scatter_size=100, scatter_alpha=0.05,
    stream_density=1.2,
    streamline_thickness=3.0,
    arrowsize=1.0
)
flowmap_results["dentate_gyrus"] = emb_dg

In [ ]:
# dentate gyrus
emb_dg = flowmap_plot_single(
    "larry",
    DATASETS["larry"],
    scatter_size=50, scatter_alpha=0.01,
    stream_density=1.3,
    streamline_thickness=4.0,
    arrowsize=1.5
)
flowmap_results["dentate_gyrus"] = emb_dg